# Enterprise Data Architecture & Reconciliation — Completed Pipeline

## The Scenario
You are a Senior Data Analyst at a FinTech company. The CFO needs a Q1–Q4 Financial Audit Dashboard.
You have three messy data sources to combine:

1. **HR Database** (`enterprise_database.db`) — a SQL database where users have multiple historical rows; we only want each user's *most recent* status.
2. **Server Logs** (`server_logs.txt`) — 100,000 raw text log lines with transaction data hidden inside a JSON-like payload string.
3. **Exchange Rates** (`daily_exchange_rates.csv`) — daily EUR→USD rates, missing on weekends (markets are closed).

**Goal:** build one clean pipeline that produces `power_bi_ready_dataset.csv`, a single flat table ready to plug into Power BI.

> **Beginner note:** each phase below has a short plain-English explanation *before* the code, so you understand *why* we're doing each step, not just *what* the code does.

## Phase 1 — SQL Extraction: Finding Currently "Active" Users

**The problem:** the `users` table logs every status change a user ever had. So `U-00001` might appear 3 times: `Pending`, `Active`, `Suspended`. If we don't handle this, we'll double- or triple-count that user, and we might treat someone as "Active" even though they were later suspended.

**The fix — a SQL "window function":**
`ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC)` numbers each user's rows from newest (1) to oldest, *per user* (that's what "partition by" means — restart the numbering for every new user_id). We then keep only `rn = 1` (the newest row) and filter for `status = 'Active'`.

This gives us one row per user — their single, current status.

In [ ]:
import sqlite3
import pandas as pd
import re
import time
import matplotlib.pyplot as plt

t_start = time.time()  # we'll use this to prove the pipeline runs in under 2 minutes (Phase 4 requirement)

# 1. Connect to the database
conn = sqlite3.connect('enterprise_database.db')

# 2 & 3. Window function: rank each user's rows newest-first, keep only the newest,
#         and only keep it if that newest status is 'Active'
sql_query = """
SELECT user_id, name, status, updated_at
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC) AS rn
    FROM users
)
WHERE rn = 1 AND status = 'Active';
"""

# 4. Load straight into a DataFrame
df_active_users = pd.read_sql_query(sql_query, conn)
conn.close()

print(f"Active users found: {len(df_active_users):,}")
df_active_users.head()

## Phase 2 — Regex Parsing: Pulling Data Out of Messy Log Text

**The problem:** each log line looks like this:

`[2023-03-15 00:00:00] INFO: Transaction processed. payload={"u":"U-19484", "item_code":"PRD-874", "eur_val":3404.44}`

The Date, User ID, Product ID, and Euro value are all buried inside one long string. Some lines also say `ERROR` instead of `INFO`, and those use a different field (`failed_val`) because the transaction failed — we must drop those.

**The fix — one regex with "named groups":**
A regex (regular expression) is a pattern that describes text to search for. `(?P<Date>\d{4}-\d{2}-\d{2})` means "capture 4 digits, a dash, 2 digits, a dash, 2 digits, and remember it under the name `Date`". We do the same for the User ID, Product ID, and Euro value.

**Why this also solves the ERROR problem for free:** our pattern specifically looks for `"eur_val":`. `ERROR` lines don't have that field — they have `"failed_val"` instead — so the regex simply won't match those lines at all. When we later drop rows where the match failed, the ERROR rows disappear automatically.

**Why it's fast (vectorized):** `.str.extract()` applies the regex to all 100,000 rows at once, in one call — no Python `for` loop needed.

In [ ]:
# 1. Load the raw logs (it's actually a proper CSV under the hood, just with a .txt extension)
logs = pd.read_csv('server_logs.txt')

# 2 & 3. One vectorized regex pulls out all 4 fields at once, for all rows simultaneously
log_pattern = (
    r'\[(?P<Date>\d{4}-\d{2}-\d{2}) [\d:]+\]\s+(?P<Level>\w+):'
    r'.*?"u":"(?P<User_ID>U-\d+)",\s*"item_code":"(?P<Product_ID>PRD-\d+)",\s*"eur_val":(?P<EUR_Value>[\d.]+)'
)
extracted = logs['Raw_Server_Log'].str.extract(log_pattern)
extracted['EUR_Value'] = pd.to_numeric(extracted['EUR_Value'], errors='coerce')

# Rows that didn't match (ERRORs, or malformed lines) come back as NaN — drop them
extracted_data = extracted.dropna(subset=['EUR_Value']).drop(columns=['Level']).reset_index(drop=True)
extracted_data['Date'] = pd.to_datetime(extracted_data['Date'])

print(f"Raw log lines: {len(logs):,}")
print(f"Valid transactions extracted (ERRORs filtered out): {len(extracted_data):,}")
extracted_data.head()

## Phase 3 — Temporal Merging: Exchange Rates + Users + Revenue

Three steps here:

1. **Fill in weekend exchange rates.** Markets are closed Saturday/Sunday, so those rows are blank (`NaN`). `.ffill()` ("forward fill") copies the *last known value downward* — so Friday's rate rolls into Saturday and Sunday. This only works if the data is sorted by date first.
2. **Join transactions → exchange rate → active users.** We merge the transaction logs with the exchange rates by `Date`, then merge that with our Phase 1 active-users table by `User_ID`. Using an *inner* join on the users step means only transactions belonging to a currently-Active user survive — exactly what the CFO wants for this audit.
3. **Calculate USD Revenue** = Euro Value × Exchange Rate.

In [ ]:
# 1. Load exchange rates and forward-fill the weekend gaps
rates = pd.read_csv('daily_exchange_rates.csv')
rates['Date'] = pd.to_datetime(rates['Date'])
rates = rates.sort_values('Date')                      # ffill only works correctly on sorted dates
rates['EUR_to_USD'] = rates['EUR_to_USD'].ffill()

print("Missing rates remaining after ffill:", rates['EUR_to_USD'].isna().sum())

# 2. Merge: transactions -> exchange rates (by Date)
merged = extracted_data.merge(rates, on='Date', how='left')

# 2b. Merge: -> active users (by User ID). inner join keeps only currently-Active users.
merged = merged.merge(
    df_active_users[['user_id', 'name']],
    left_on='User_ID', right_on='user_id', how='inner'
)

# 3. Revenue calculation (fully vectorized — no loops)
merged['USD_Revenue'] = merged['EUR_Value'] * merged['EUR_to_USD']

df_final = merged[['Date', 'User_ID', 'name', 'Product_ID', 'EUR_Value', 'EUR_to_USD', 'USD_Revenue']]
df_final = df_final.rename(columns={'name': 'Customer_Name', 'EUR_to_USD': 'Exchange_Rate'})

print(f"Final merged rows: {len(df_final):,}")
df_final.head()

## Phase 4 — Aggregation, Visualization & Efficiency Check

Now we summarize `df_final` two ways the CFO asked for:
- **Total USD Revenue by Month** (a trend line)
- **Top 5 Customers by total USD spent** (a bar chart)

Both `.groupby()` calls below are vectorized (pandas does this in optimized C code under the hood) — this is what "no `for` loops" means in practice. We'll also print the total time the whole notebook took, to prove we're well under the 2-minute limit.

In [ ]:
# Monthly revenue trend
monthly = df_final.copy()
monthly['Month'] = monthly['Date'].dt.to_period('M').astype(str)
monthly_revenue = monthly.groupby('Month')['USD_Revenue'].sum().reset_index()

# Top 5 customers by total revenue
top_customers = (
    df_final.groupby(['User_ID', 'Customer_Name'])['USD_Revenue']
    .sum()
    .reset_index()
    .sort_values('USD_Revenue', ascending=False)
    .head(5)
)

print("Monthly revenue:")
print(monthly_revenue)
print("\nTop 5 customers:")
print(top_customers)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(monthly_revenue['Month'], monthly_revenue['USD_Revenue'], marker='o', color='steelblue')
axes[0].set_title('Total USD Revenue by Month')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('USD Revenue')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(top_customers['Customer_Name'], top_customers['USD_Revenue'], color='darkorange')
axes[1].set_title('Top 5 Customers by Revenue')
axes[1].set_xlabel('Customer')
axes[1].set_ylabel('USD Revenue')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## Phase 5 — BI Handoff: Export for Power BI

The last step is exporting `df_final` to a clean CSV that Power BI will use as its data source. We also print the total elapsed time to confirm we're comfortably under the 2-minute / no-loops requirement.

In [ ]:
df_final.to_csv('power_bi_ready_dataset.csv', index=False)

elapsed = time.time() - t_start
print(f"power_bi_ready_dataset.csv saved: {len(df_final):,} rows, {df_final.shape[1]} columns")
print(f"Total pipeline runtime: {elapsed:.2f} seconds (limit was 120 seconds)")

---
## Next Step: Power BI Dashboard

`power_bi_ready_dataset.csv` is now ready. Open it in **Power BI Desktop** to build:
- A Date Slicer
- A DAX measure for Total USD Revenue
- A Line Chart of revenue over time
- A Bar Chart of the Top 5 customers

(See the separate step-by-step Power BI guide for exactly how to build each of these.)